# Сдвинутый экспоненциальный шум: точный фильтр против гауссовской аппроксимации и ЦПТ

$\theta$ имеет четыре состояния, а $Y=(Y_1,Y_2)$ задаётся на той же сетке и
с теми же параметрами траектории, что в Pareto-аналоге. Seed равен
`20260803`.

Для точного канала

$$X = Y_1 + Y_2\varepsilon, \qquad
\varepsilon = E + 1 - \frac{1}{\sqrt{12}}, \qquad
E \sim \operatorname{Exp}(\sqrt{12}).$$

Поэтому $E\varepsilon=1$, $\operatorname{Var}\varepsilon=1/12$ и

$$E[X\mid Y]=Y_1+Y_2, \qquad
\operatorname{Var}(X\mid Y)=Y_2^2/12.$$

Точный фильтр использует плотность с носителем
$x \ge Y_1 + Y_2(1-1/\sqrt{12})$; гауссовские фильтры используют только
два указанных условных момента.

## Постановка сравнения

Генерируются одна траектория и один поток точных наблюдений с шагом $h_t=1$.
На них сравниваются три фильтра:

| Фильтр | конфиг | вход |
|---|---|---|
| точный | `exponential_obs` | исходные наблюдения, $h_t=1$ |
| гауссовский по сумме | `exponential_obs_approx` | суммы полных блоков по 10, $h_t=10$ |
| ЦПТ по среднему | `exponential_obs_clt` | средние тех же блоков, $h_t=10$ |

Последний неполный блок, если он возникает, отбрасывается. Новая выборка для
аппроксимаций не генерируется. Сумма и среднее одного блока являются
эквивалентными представлениями: в `_clt` условные drift и var масштабированы
на $1/10$ и $1/10^2$ соответственно.

Оценки точного фильтра прореживаются до границ блоков. RMSE всех трёх
вариантов считается на общей сетке $h_t=10$ относительно одной скрытой
траектории; затем строятся графики вероятностей состояний и оценок обеих
координат $Y$.

Ноутбук намеренно сохранён без выполнения: полный точный прогон содержит
86400 узлов и 86399 обновлений Filter.update и может занимать много времени.


In [ ]:
import _bootstrap  # noqa: F401

import numpy as np
import matplotlib.pyplot as plt

from discretized_filter.config import set_config
from discretized_filter.core.smjp import sparse_mc
from discretized_filter.core.filter import Filter
from discretized_filter.utils.grids import to_discrete
from discretized_filter.visualization.plots import (
    plot_theta_background, theta_labels_default, theta_colors_default,
)

cfg_approx = set_config('exponential_obs_approx')
cfg_clt = set_config('exponential_obs_clt')
cfg = set_config('exponential_obs')      # точный, активен последним

print(f'точный  : N={cfg.N}, M={cfg.M}, K={cfg.K}, ht={cfg.ht}, '
      f'узлов={cfg.t_net_filtering.shape[0]}, узлов сетки={cfg.M_net.shape[1]}')
print(f'аппрокс.: N={cfg_approx.N}, M={cfg_approx.M}, K={cfg_approx.K}, '
      f'ht={cfg_approx.ht}, узлов={cfg_approx.t_net_filtering.shape[0]}, '
      f'узлов сетки={cfg_approx.M_net.shape[1]}')
print(f'ЦПТ-сред.: N={cfg_clt.N}, M={cfg_clt.M}, K={cfg_clt.K}, '
      f'ht={cfg_clt.ht}, узлов={cfg_clt.t_net_filtering.shape[0]}, '
      f'узлов сетки={cfg_clt.M_net.shape[1]}')

In [ ]:
theta, y, t = sparse_mc(cfg.p0, cfg.Lambda, cfg.lam, cfg.T, cfg.get_y, cfg.y_intervals)
obs = cfg.get_obs(cfg.t_net_filtering, theta, y, t)

# наблюдения для фильтров с ht=10 -- не новая генерация, а суммирование
# блоков по 10 из ОДНОЙ и той же выборки obs (наблюдение -- процесс с
# приращениями, см. markdown выше, 'Почему агрегирование корректно')
ratio = int(round(cfg_approx.ht / cfg.ht))          # 10
n_blocks = obs.shape[0] // ratio
n_dropped = obs.shape[0] - n_blocks * ratio
obs_agg = obs[:n_blocks * ratio].reshape(n_blocks, ratio, cfg.K).sum(axis=1)
assert n_blocks == cfg_approx.t_net_filtering.shape[0] - 1, (
    f'{n_blocks} блоков наблюдений против '
    f'{cfg_approx.t_net_filtering.shape[0] - 1} шагов аппроксимационного '
    'фильтра -- сетки конфигов разъехались (T/seed/N/Lambda/y_intervals/num1 '
    'должны совпадать у exponential_obs и exponential_obs_approx)'
)

# то же самое блочное окно, только осреднённое (а не просуммированное) --
# наблюдения для третьего (ЦПТ-осредняющего) фильтра exponential_obs_clt
obs_mean = obs_agg / ratio

print(f'скачков theta: {len(t)}')
print(f'отброшено наблюдений неполного хвоста: {n_dropped}')
header = 'выборка            ht        n       mean        std      max|x|'
print(header)
for label, ht_, o in [
    ('точная', cfg.ht, obs),
    ('агрег. (сумма 10)', cfg_approx.ht, obs_agg),
    ('ЦПТ (среднее 10)', cfg_clt.ht, obs_mean),
]:
    print(f'{label:<18} {ht_:5.0f} {o.shape[0]:7d} {o.mean():9.3f} '
          f'{o.std():9.3f} {np.abs(o).max():9.3f}')

In [ ]:
def run(config, observations):
    f = Filter(
        config.pi_init, config.pi, config.M_net, config.C,
        config.N, config.Lambda, config.ht, config.delta, config.obs_density,
        n_points=config.n_points, two_jumps=config.two_jumps,
        filter_step=config.filter_step,
    )
    est = f.estimate()
    th, yy = [est[0]], [est[1]]
    for obs_ in observations:
        f.update(obs_)
        est = f.estimate()
        th.append(est[0])
        yy.append(est[1])
    return np.array(th), np.array(yy)


# ВНИМАНИЕ О СТОИМОСТИ: при T=24*3600, ht=1 сетка содержит 86400 узлов,
# а точный фильтр выполняет 86399 обновлений Filter.update; на каждом обновлении
# ядро оценивает плотность порядка N^2 * n_grid^2 * n_points
# = 16 * 400^2 * 2 ~ 5e6 раз -- полный прогон занимает часы. Оба фильтра с
# ht=10 (сумма и ЦПТ-среднее) дешевле примерно в 10 раз по числу обновлений и делают
# ОДИНАКОВОЕ число шагов друг с другом, поэтому третий прогон почти не
# добавляет стоимости ко второму. Для пробного прогона уменьшите T во ВСЕХ
# трёх конфигах (configs/exponential_obs.py, configs/exponential_obs_approx.py,
# configs/exponential_obs_clt.py, например до 3*3600) или num1, перегенерировав
# конфиги и траекторию заново.
runs = {
    'точный, ht=1': run(cfg, obs),
    'аппрокс., ht=10': run(cfg_approx, obs_agg),
    'ЦПТ-среднее, ht=10': run(cfg_clt, obs_mean),
}

In [ ]:
th_exact, yy_exact = runs['точный, ht=1']
th_approx, yy_approx = runs['аппрокс., ht=10']
th_clt, yy_clt = runs['ЦПТ-среднее, ht=10']

# общая сетка сравнения -- ht=10 (сетка аппроксимационных фильтров);
# оценки точного фильтра (своя сетка ht=1) прореживаем с шагом ratio
dtheta = to_discrete(
    np.vstack([np.int64(theta == i) for i in range(cfg.N)]).T, t, cfg.T, cfg_approx.ht
)
dY = to_discrete(y, t, cfg.T, cfg_approx.ht)
n = dtheta.shape[0]

th_exact_thin = th_exact[::ratio]
yy_exact_thin = yy_exact[::ratio]

header = 'фильтр                    RMSE theta   RMSE Y1   RMSE Y2'
print(header)
for label, th, yy in [
    ('точный (прореж., ht=10)', th_exact_thin, yy_exact_thin),
    ('аппрокс., ht=10', th_approx, yy_approx),
    ('ЦПТ-среднее, ht=10', th_clt, yy_clt),
]:
    rt = np.sqrt(((dtheta - th[:n]) ** 2).mean())
    ry1 = np.sqrt(((dY[:, 0] - yy[:n, 0]) ** 2).mean())
    ry2 = np.sqrt(((dY[:, 1] - yy[:n, 1]) ** 2).mean())
    print(f'{label:<24} {rt:11.4f} {ry1:9.4f} {ry2:9.4f}')

# справочно: RMSE точного фильтра на ЕГО СОБСТВЕННОЙ сетке ht=1 (без
# прореживания) -- показывает, чего лишается точный фильтр при огрублении
# сетки сравнения до ht=10
dtheta1 = to_discrete(
    np.vstack([np.int64(theta == i) for i in range(cfg.N)]).T, t, cfg.T, cfg.ht
)
dY1 = to_discrete(y, t, cfg.T, cfg.ht)
n1 = dtheta1.shape[0]
rt1 = np.sqrt(((dtheta1 - th_exact[:n1]) ** 2).mean())
ry1_1 = np.sqrt(((dY1[:, 0] - yy_exact[:n1, 0]) ** 2).mean())
ry2_1 = np.sqrt(((dY1[:, 1] - yy_exact[:n1, 1]) ** 2).mean())
label1 = 'точный (своя сетка ht=1)'
print(f'{label1:<24} {rt1:11.4f} {ry1_1:9.4f} {ry2_1:9.4f}'
      '   <- справочно, другая сетка сравнения')

In [ ]:
styles = {
    'точный, ht=1': dict(color='tab:red', lw=1.0),
    'аппрокс., ht=10': dict(color='tab:blue', lw=1.4),
    # пунктир: тождественен предыдущей кривой (см. markdown), иначе они
    # неразличимы на графике
    'ЦПТ-среднее, ht=10': dict(color='tab:green', lw=1.4, ls='--'),
}

fig, axes = plt.subplots(cfg.N, 1, figsize=(13, 9), layout='constrained', sharex=True)
for n_, ax in enumerate(axes):
    plot_theta_background(
        ax, theta, t, theta_labels_default, theta_colors_default, 1, alpha=0.15
    )
    # каждый фильтр -- по своей временной сетке (ht=1 и ht=10), без подгонки
    ax.plot(cfg.t_net_filtering, th_exact[:, n_],
            label='точный, ht=1', **styles['точный, ht=1'])
    ax.plot(cfg_approx.t_net_filtering, th_approx[:, n_],
            label='аппрокс., ht=10', **styles['аппрокс., ht=10'])
    ax.plot(cfg_clt.t_net_filtering, th_clt[:, n_],
            label='ЦПТ-среднее, ht=10', **styles['ЦПТ-среднее, ht=10'])
    ax.set(ylabel=rf'$\hat\theta^{n_ + 1}_t$', ylim=(-0.05, 1.05))
axes[-1].set(xlabel='$t$', xlim=(0, cfg.T))
axes[0].legend(ncol=3, fontsize=8, loc='upper right')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), layout='constrained', sharex=True)
labels_y = ['$Y^1_t$', '$Y^2_t$']
for j, ax in enumerate(axes):
    ax.step([0] + list(t), [y[0, j]] + list(y[:, j]),
            where='pre', color='k', lw=2.5, alpha=0.35, label=labels_y[j])
    ax.plot(cfg.t_net_filtering, yy_exact[:, j],
            label='точный, ht=1', **styles['точный, ht=1'])
    ax.plot(cfg_approx.t_net_filtering, yy_approx[:, j],
            label='аппрокс., ht=10', **styles['аппрокс., ht=10'])
    ax.plot(cfg_clt.t_net_filtering, yy_clt[:, j],
            label='ЦПТ-среднее, ht=10', **styles['ЦПТ-среднее, ht=10'])
    ax.set(ylabel=rf'$\hat Y^{j + 1}_t$')
axes[-1].set(xlabel='$t$', xlim=(0, cfg.T))
axes[0].legend(ncol=4, fontsize=9, loc='lower right')
plt.show()

In [ ]:
# итог: разность точный (прореженный до ht=10) минус аппроксимационный на
# общей сетке ht=10
d_theta = th_exact_thin[:n] - th_approx[:n]
d_Y = yy_exact_thin[:n] - yy_approx[:n]

print(f'макс |точный(проред.) - аппрокс.| по theta = {np.abs(d_theta).max():.4f}')
print(f'макс |точный(проред.) - аппрокс.| по Y     = {np.abs(d_Y).max():.4f}')